In [5]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, confusion_matrix

# ---------------- USER SETTINGS ----------------
DATA_PATH = "data/soybeans_2016_2023.parquet"   # change crop file here
TARGET_COL = "yield"                       # regression target
GROUP_COL  = "farm_name"                   # group split key
N_SPLITS   = 5

# feature choices
INCLUDE_YEAR_LAT = True                    # you said: include Year + latitude
USE_LONGITUDE = False                      # you decided to drop longitude
# ------------------------------------------------

# Load
df = pd.read_parquet(DATA_PATH)

# Drop rows with missing target
df = df[df[TARGET_COL].notna()].copy()

# Clean: drop old wrong normalized cols if present
bad_norm = [c for c in ["normalized_yield_pct", "normalized_yield_z"] if c in df.columns]

# Clean: remove SAR duplicates ending with _2
sar_dup2 = [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and c.endswith("_2")]

df = df.drop(columns=bad_norm + sar_dup2, errors="ignore")

print("Rows:", len(df), "Cols:", len(df.columns))
print("Dropped:", bad_norm, "and", len(sar_dup2), "SAR _2 cols")

# Continuous target (float)
y = df[TARGET_COL].to_numpy(dtype=np.float32)

# Groups
groups = df[GROUP_COL].astype(str).to_numpy()
print("Unique groups:", len(np.unique(groups)))

# CV folds (fixed across models)
gkf = GroupKFold(n_splits=N_SPLITS)
folds = list(gkf.split(np.zeros(len(df)), y, groups=groups))
print("Built folds:", len(folds))

# ------------ Metrics helpers ------------
def regression_metrics(y_true, y_pred):
    """Return R², RMSE, MAE."""
    r2 = r2_score(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    return r2, rmse, mae

def zone_thresholds_from_train(y_train, q_low=0.4, q_high=0.6):
    """Compute yield thresholds from TRAIN only (prevents leakage)."""
    low = np.quantile(y_train, q_low)
    high = np.quantile(y_train, q_high)
    return low, high

def yield_to_zone(y_vals, low, high):
    """Convert continuous yield to zones using thresholds."""
    # 0=low, 1=medium, 2=high
    z = np.zeros_like(y_vals, dtype=np.int64)
    z[(y_vals >= low) & (y_vals <= high)] = 1
    z[y_vals > high] = 2
    return z

def zone_metrics(y_true, y_pred, y_train_for_thresholds):
    """
    Evaluate 'zone accuracy' + confusion matrix for regression predictions.
    Thresholds are computed from TRAIN yields only.
    """
    low, high = zone_thresholds_from_train(y_train_for_thresholds, 0.4, 0.6)
    z_true = yield_to_zone(y_true, low, high)
    z_pred = yield_to_zone(y_pred, low, high)
    acc = (z_true == z_pred).mean()
    cm = confusion_matrix(z_true, z_pred, labels=[0,1,2])
    return float(acc), cm

def summarize_cv_regression(r2s, rmses, maes, name="MODEL"):
    print(f"\n{name} CV Summary (regression)")
    print(f"R²:   {np.mean(r2s):.3f} ± {np.std(r2s):.3f}")
    print(f"RMSE: {np.mean(rmses):.3f} ± {np.std(rmses):.3f}")
    print(f"MAE:  {np.mean(maes):.3f} ± {np.std(maes):.3f}")

def summarize_cv_zones(zone_accs, cm_sum, name="MODEL"):
    print(f"\n{name} CV Summary (zone-from-regression)")
    print(f"Zone Accuracy: {np.mean(zone_accs):.3f} ± {np.std(zone_accs):.3f}")
    print("Zone confusion matrix sum (rows=true low/med/high, cols=pred):\n", cm_sum)

def build_year_mean_map(years_train, y_train_raw):
    """TRAIN-only map: year -> mean yield in that year."""
    year_means = {}
    for yr in np.unique(years_train):
        vals = y_train_raw[years_train == yr]
        year_means[int(yr)] = float(np.mean(vals))
    return year_means

def lookup_year_means(years, year_means, fallback):
    """Per-sample mean yield by year using TRAIN-derived map."""
    out = np.empty(len(years), dtype=np.float32)
    for i, yr in enumerate(years):
        out[i] = year_means.get(int(yr), fallback)
    return out

def zone_metrics_pm10_from_ratio(y_true_ratio, y_pred_ratio):
    """Zones in ratio space: low<0.9, med 0.9-1.1, high>1.1."""
    z_true = np.where(y_true_ratio < 0.9, 0, np.where(y_true_ratio <= 1.1, 1, 2))
    z_pred = np.where(y_pred_ratio < 0.9, 0, np.where(y_pred_ratio <= 1.1, 1, 2))
    acc = float((z_true == z_pred).mean())
    cm = confusion_matrix(z_true, z_pred, labels=[0,1,2])
    return acc, cm


Rows: 694250 Cols: 349
Dropped: ['normalized_yield_pct', 'normalized_yield_z'] and 20 SAR _2 cols
Unique groups: 125
Built folds: 5


# Gradient Boosting baseline (XGBoost-like without xgboost)
# What / Why

HistGradientBoostingClassifier is the best “xgboost-ish” model you can run with your installed libs.

Preprocessing

Median impute

No scaling required

Yield normalization strategy

We model crop yield as a relative deviation from the annual mean yield, rather than as an absolute value.
For each cross-validation fold, we compute the mean yield per crop–year using only the training data.
The regression target is defined as:

𝑦
norm
=
𝑦
𝜇
train
(
crop
,
year
)
y
norm
	​

=
μ
train
	​

(crop,year)
y
	​


where 
𝑦
y is the observed yield and 
𝜇
train
μ
train
	​

 is the mean yield for that crop and year computed exclusively from the training set.

During evaluation, predicted normalized yields are converted back to physical units:

𝑦
^
=
𝑦
^
norm
×
𝜇
test
(
crop
,
year
)
y
^
	​

=
y
^
	​

norm
	​

×μ
test
	​

(crop,year)

Model performance is evaluated using continuous regression metrics (R², RMSE, MAE).
Additionally, predictions are mapped to agronomically meaningful yield zones (low / medium / high) using fixed thresholds (0.9× and 1.1× the annual mean), enabling direct comparison between continuous and categorical modeling approaches.

This procedure avoids data leakage, preserves physical interpretability, and removes inter-annual yield scale effects.

2️⃣ Why this normalization is the right one (conceptually)

You are doing ratio normalization, not z-scoring.

What this removes

Technological trend (better seeds, management)

Climate-driven year-to-year yield scale shifts

Crop-specific magnitude differences

What it preserves

Within-year spatial variation

Relative productivity patterns

Physical meaning of prediction errors

Why not z-score?

Z-score destroys interpretability

Z-score mixes variance structure across years

Z-score breaks agronomic thresholds

Your advisor’s 0.9× / 1.1× rule only makes sense in ratio space.

3️⃣ Yield zones: how you explain them clearly

Yield zones are defined relative to the annual mean yield rather than by global percentiles.
For a given crop and year:

Low yield: 
𝑦
<
0.9
×
𝜇
y<0.9×μ

Medium yield: 
0.9
×
𝜇
≤
𝑦
≤
1.1
×
𝜇
0.9×μ≤y≤1.1×μ

High yield: 
𝑦
>
1.1
×
𝜇
y>1.1×μ

When training regression models, zone labels are not used during training.
Instead, predicted yields are mapped to zones post-hoc for evaluation, ensuring fair comparison between continuous and categorical approaches.

That last sentence is important — it shows methodological rigor.

In [2]:
import os, time
import numpy as np

# --- Encourage sklearn/OpenMP to use your allocated cores (best effort) ---
# (Works if your sklearn build uses OpenMP / MKL underneath)
os.environ["OMP_NUM_THREADS"] = "50"
os.environ["MKL_NUM_THREADS"] = "50"
os.environ["OPENBLAS_NUM_THREADS"] = "50"
os.environ["NUMEXPR_NUM_THREADS"] = "50"

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer

def fmt_time(seconds: float) -> str:
    seconds = max(0.0, float(seconds))
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:d}h {m:02d}m {s:02d}s" if h > 0 else f"{m:02d}m {s:02d}s"

# ---------------- Features ----------------
SAR_COLS = [c for c in df.columns if c.startswith("VV_") or c.startswith("VH_")]
STATIC_COLS = ["Year", "latitude"]   # include these; no longitude

X = df[SAR_COLS + STATIC_COLS].to_numpy(dtype=np.float32)
y = df["yield"].to_numpy(dtype=np.float32)

# ---------------- CV evaluation containers ----------------
r2s, rmses, maes = [], [], []
zone_accs = []
cm_zone_sum = np.zeros((3, 3), dtype=int)

fold_times = []
overall_start = time.time()
n_folds = len(folds)

for k, (tr, te) in enumerate(folds, start=1):
    fold_start = time.time()

    X_train_raw, X_test_raw = X[tr], X[te]

    # ---- Train-only imputation (prevents leakage) ----
    imp = SimpleImputer(strategy="median")
    X_train = imp.fit_transform(X_train_raw)
    X_test  = imp.transform(X_test_raw)

    # ---- Raw yields + years ----
    y_train_raw = y[tr].astype(np.float32)
    y_test_raw  = y[te].astype(np.float32)

    years_train = df["Year"].to_numpy()[tr]
    years_test  = df["Year"].to_numpy()[te]

    # ---- TRAIN-only year means ----
    year_means = build_year_mean_map(years_train, y_train_raw)
    fallback = float(np.mean(y_train_raw))

    mu_train = lookup_year_means(years_train, year_means, fallback)
    mu_test  = lookup_year_means(years_test,  year_means, fallback)

    # ---- Normalized targets for training ----
    y_train = (y_train_raw / mu_train).astype(np.float32)
    y_test_ratio = (y_test_raw / mu_test).astype(np.float32)

    # ---- Model (CPU) ----
    model = HistGradientBoostingRegressor(
        max_depth=8,
        learning_rate=0.07,
        max_iter=300,
        random_state=42
    )

    model.fit(X_train, y_train)
    pred_ratio = model.predict(X_test).astype(np.float32)

    # ---- Back to raw for metrics ----
    pred_raw = pred_ratio * mu_test

    r2, rmse, mae = regression_metrics(y_test_raw, pred_raw)
    zacc, cmz = zone_metrics_pm10_from_ratio(y_test_ratio, pred_ratio)

    r2s.append(r2); rmses.append(rmse); maes.append(mae)
    zone_accs.append(zacc); cm_zone_sum += cmz

    fold_time = time.time() - fold_start
    fold_times.append(fold_time)

    avg_fold = float(np.mean(fold_times))
    eta_remaining = (n_folds - k) * avg_fold
    elapsed = time.time() - overall_start

    print(
        f"Fold {k}/{n_folds}: "
        f"R²={r2:.3f} | RMSE={rmse:.3f} | MAE={mae:.3f} || zone_acc={zacc:.3f} "
        f"| fold_time={fmt_time(fold_time)} | elapsed={fmt_time(elapsed)} | ETA={fmt_time(eta_remaining)}"
    )


summarize_cv_regression(r2s, rmses, maes, name="HistGradientBoostingRegressor (SAR + Year + latitude)")
summarize_cv_zones(zone_accs, cm_zone_sum, name="HistGradientBoostingRegressor (zone eval)")
print("Total wall time:", fmt_time(time.time() - overall_start))


Fold 1/5: R²=0.445 | RMSE=34.943 | MAE=26.600 || zone_acc=0.555 | fold_time=01m 32s | elapsed=01m 32s | ETA=06m 11s
Fold 2/5: R²=0.450 | RMSE=35.557 | MAE=26.226 || zone_acc=0.590 | fold_time=01m 51s | elapsed=03m 24s | ETA=05m 06s
Fold 3/5: R²=0.389 | RMSE=37.297 | MAE=29.542 || zone_acc=0.523 | fold_time=01m 25s | elapsed=04m 49s | ETA=03m 13s
Fold 4/5: R²=0.437 | RMSE=36.435 | MAE=26.875 || zone_acc=0.589 | fold_time=01m 55s | elapsed=06m 44s | ETA=01m 41s
Fold 5/5: R²=0.413 | RMSE=33.657 | MAE=26.152 || zone_acc=0.521 | fold_time=02m 45s | elapsed=09m 29s | ETA=00m 00s

HistGradientBoostingRegressor (SAR + Year + latitude) CV Summary (regression)
R²:   0.427 ± 0.023
RMSE: 35.578 ± 1.248
MAE:  27.079 ± 1.259

HistGradientBoostingRegressor (zone eval) CV Summary (zone-from-regression)
Zone Accuracy: 0.556 ± 0.030
Zone confusion matrix sum (rows=true low/med/high, cols=pred):
 [[433146 209889  28487]
 [151548 326200 103736]
 [ 45316 398277 412397]]
Total wall time: 09m 29s


Fold 1/5: R²=0.445 | RMSE=34.943 | MAE=26.600 || zone_acc=0.555 | fold_time=01m 32s | elapsed=01m 32s | ETA=06m 11s
Fold 2/5: R²=0.450 | RMSE=35.557 | MAE=26.226 || zone_acc=0.590 | fold_time=01m 51s | elapsed=03m 24s | ETA=05m 06s
Fold 3/5: R²=0.389 | RMSE=37.297 | MAE=29.542 || zone_acc=0.523 | fold_time=01m 25s | elapsed=04m 49s | ETA=03m 13s
Fold 4/5: R²=0.437 | RMSE=36.435 | MAE=26.875 || zone_acc=0.589 | fold_time=01m 55s | elapsed=06m 44s | ETA=01m 41s
Fold 5/5: R²=0.413 | RMSE=33.657 | MAE=26.152 || zone_acc=0.521 | fold_time=02m 45s | elapsed=09m 29s | ETA=00m 00s

HistGradientBoostingRegressor (SAR + Year + latitude) CV Summary (regression)
R²:   0.427 ± 0.023
RMSE: 35.578 ± 1.248
MAE:  27.079 ± 1.259

HistGradientBoostingRegressor (zone eval) CV Summary (zone-from-regression)
Zone Accuracy: 0.556 ± 0.030
Zone confusion matrix sum (rows=true low/med/high, cols=pred):
 [[433146 209889  28487]
 [151548 326200 103736]
 [ 45316 398277 412397]]
Total wall time: 09m 29s

 The Regression Performance ($R^2 \approx 0.427$)The Meaning: An $R^2$ of 0.427 means your model (using only SAR, Year, and Latitude) explains about 43% of the variance in corn yields.The Opinion: In agricultural remote sensing, especially using only radar (SAR) without optical data or soil info, this is actually a very respectable baseline. Radar can see through clouds, but it struggles with "saturation" (where it can't tell the difference between a big plant and a massive plant).Consistency: Your $R^2$ is stable across folds ($0.38$ to $0.45$), meaning your model isn't overfitting to specific farms. This is a good sign that your GroupKFold is working correctly.2. The Zone Accuracy ($\approx 55.6\%$)The Meaning: When you convert your predictions into "Low, Medium, High" categories using your $\pm 10\%$ rule, you are right 55.6% of the time.The Opinion: Random guessing would give you 33.3% accuracy. You are significantly better than random. However, look at your Confusion Matrix:The Good: The model is best at identifying the "Low" (433k) and "High" (412k) zones.The "Confusion": Look at the High-to-Low and Low-to-High errors ($28,487$ and $45,316$). These are "extreme errors." They are relatively low compared to the diagonal, which means the model rarely confuses a "High" yield for a "Low" yield.The "Middle" Problem: The model often predicts "Medium" when the reality is "High" ($398,277$ cases). This suggests the model is "conservative"—it tends to predict closer to the average.3. Training Speed & ResourcesWall Time: 9 minutes for 5 folds is excellent.Resource Use: You are using 50 cores on an H200 node. Since HistGradientBoosting is a CPU-based algorithm in sklearn, you aren't actually using the H200 GPU yet.The "Why": Standard Sklearn doesn't talk to GPUs. To use the H200 for this part, we would need to switch to XGBoost or LightGBM with the device="cuda" parameter.4. Technical Observations (Teaching Moment)Target Scaling: You trained on the ratio ($yield / mean\_year$). This was a smart move! It focuses the model on "relative performance" rather than absolute bushels.Imputation: Using SimpleImputer(strategy="median") is safe, but for SAR time-series, "Median" might blur the growth curve. If a date is missing, it might be better to interpolate between the previous and next date, but for a Gradient Boosting model, median imputation is a standard first step.Leakage Prevention: You correctly computed the year_means and thresholds using only the training set for each fold. This is "Gold Standard" research—it ensures your results aren't inflated by knowing the future or the "global" average.

# Wheat

In [4]:
import os, time
import numpy as np

# --- Encourage sklearn/OpenMP to use your allocated cores (best effort) ---
# (Works if your sklearn build uses OpenMP / MKL underneath)
os.environ["OMP_NUM_THREADS"] = "50"
os.environ["MKL_NUM_THREADS"] = "50"
os.environ["OPENBLAS_NUM_THREADS"] = "50"
os.environ["NUMEXPR_NUM_THREADS"] = "50"

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer

def fmt_time(seconds: float) -> str:
    seconds = max(0.0, float(seconds))
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:d}h {m:02d}m {s:02d}s" if h > 0 else f"{m:02d}m {s:02d}s"

# ---------------- Features ----------------
SAR_COLS = [c for c in df.columns if c.startswith("VV_") or c.startswith("VH_")]
STATIC_COLS = ["Year", "latitude"]   # include these; no longitude

X = df[SAR_COLS + STATIC_COLS].to_numpy(dtype=np.float32)
y = df["yield"].to_numpy(dtype=np.float32)

# ---------------- CV evaluation containers ----------------
r2s, rmses, maes = [], [], []
zone_accs = []
cm_zone_sum = np.zeros((3, 3), dtype=int)

fold_times = []
overall_start = time.time()
n_folds = len(folds)

for k, (tr, te) in enumerate(folds, start=1):
    fold_start = time.time()

    X_train_raw, X_test_raw = X[tr], X[te]

    # ---- Train-only imputation (prevents leakage) ----
    imp = SimpleImputer(strategy="median")
    X_train = imp.fit_transform(X_train_raw)
    X_test  = imp.transform(X_test_raw)

    # ---- Raw yields + years ----
    y_train_raw = y[tr].astype(np.float32)
    y_test_raw  = y[te].astype(np.float32)

    years_train = df["Year"].to_numpy()[tr]
    years_test  = df["Year"].to_numpy()[te]

    # ---- TRAIN-only year means ----
    year_means = build_year_mean_map(years_train, y_train_raw)
    fallback = float(np.mean(y_train_raw))

    mu_train = lookup_year_means(years_train, year_means, fallback)
    mu_test  = lookup_year_means(years_test,  year_means, fallback)

    # ---- Normalized targets for training ----
    y_train = (y_train_raw / mu_train).astype(np.float32)
    y_test_ratio = (y_test_raw / mu_test).astype(np.float32)

    # ---- Model (CPU) ----
    model = HistGradientBoostingRegressor(
        max_depth=8,
        learning_rate=0.07,
        max_iter=300,
        random_state=42
    )

    model.fit(X_train, y_train)
    pred_ratio = model.predict(X_test).astype(np.float32)

    # ---- Back to raw for metrics ----
    pred_raw = pred_ratio * mu_test

    r2, rmse, mae = regression_metrics(y_test_raw, pred_raw)
    zacc, cmz = zone_metrics_pm10_from_ratio(y_test_ratio, pred_ratio)

    r2s.append(r2); rmses.append(rmse); maes.append(mae)
    zone_accs.append(zacc); cm_zone_sum += cmz

    fold_time = time.time() - fold_start
    fold_times.append(fold_time)

    avg_fold = float(np.mean(fold_times))
    eta_remaining = (n_folds - k) * avg_fold
    elapsed = time.time() - overall_start

    print(
        f"Fold {k}/{n_folds}: "
        f"R²={r2:.3f} | RMSE={rmse:.3f} | MAE={mae:.3f} || zone_acc={zacc:.3f} "
        f"| fold_time={fmt_time(fold_time)} | elapsed={fmt_time(elapsed)} | ETA={fmt_time(eta_remaining)}"
    )


summarize_cv_regression(r2s, rmses, maes, name="HistGradientBoostingRegressor (SAR + Year + latitude)")
summarize_cv_zones(zone_accs, cm_zone_sum, name="HistGradientBoostingRegressor (zone eval)")
print("Total wall time:", fmt_time(time.time() - overall_start))


Fold 1/5: R²=0.524 | RMSE=14.529 | MAE=11.501 || zone_acc=0.515 | fold_time=00m 44s | elapsed=00m 44s | ETA=02m 56s
Fold 2/5: R²=0.609 | RMSE=12.754 | MAE=10.082 || zone_acc=0.621 | fold_time=00m 27s | elapsed=01m 11s | ETA=01m 47s
Fold 3/5: R²=0.511 | RMSE=15.331 | MAE=10.780 || zone_acc=0.614 | fold_time=00m 14s | elapsed=01m 26s | ETA=00m 57s
Fold 4/5: R²=0.594 | RMSE=15.700 | MAE=12.009 || zone_acc=0.491 | fold_time=00m 13s | elapsed=01m 39s | ETA=00m 24s
Fold 5/5: R²=0.250 | RMSE=20.381 | MAE=14.699 || zone_acc=0.485 | fold_time=00m 17s | elapsed=01m 57s | ETA=00m 00s

HistGradientBoostingRegressor (SAR + Year + latitude) CV Summary (regression)
R²:   0.498 ± 0.130
RMSE: 15.739 ± 2.533
MAE:  11.814 ± 1.583

HistGradientBoostingRegressor (zone eval) CV Summary (zone-from-regression)
Zone Accuracy: 0.545 ± 0.060
Zone confusion matrix sum (rows=true low/med/high, cols=pred):
 [[ 93113  53897   3941]
 [ 39550 103414  20239]
 [ 20296  87656  73683]]
Total wall time: 01m 57s


# Soybeans

In [6]:
import os, time
import numpy as np

# --- Encourage sklearn/OpenMP to use your allocated cores (best effort) ---
# (Works if your sklearn build uses OpenMP / MKL underneath)
os.environ["OMP_NUM_THREADS"] = "50"
os.environ["MKL_NUM_THREADS"] = "50"
os.environ["OPENBLAS_NUM_THREADS"] = "50"
os.environ["NUMEXPR_NUM_THREADS"] = "50"

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer

def fmt_time(seconds: float) -> str:
    seconds = max(0.0, float(seconds))
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:d}h {m:02d}m {s:02d}s" if h > 0 else f"{m:02d}m {s:02d}s"

# ---------------- Features ----------------
SAR_COLS = [c for c in df.columns if c.startswith("VV_") or c.startswith("VH_")]
STATIC_COLS = ["Year", "latitude"]   # include these; no longitude

X = df[SAR_COLS + STATIC_COLS].to_numpy(dtype=np.float32)
y = df["yield"].to_numpy(dtype=np.float32)

# ---------------- CV evaluation containers ----------------
r2s, rmses, maes = [], [], []
zone_accs = []
cm_zone_sum = np.zeros((3, 3), dtype=int)

fold_times = []
overall_start = time.time()
n_folds = len(folds)

for k, (tr, te) in enumerate(folds, start=1):
    fold_start = time.time()

    X_train_raw, X_test_raw = X[tr], X[te]

    # ---- Train-only imputation (prevents leakage) ----
    imp = SimpleImputer(strategy="median")
    X_train = imp.fit_transform(X_train_raw)
    X_test  = imp.transform(X_test_raw)

    # ---- Raw yields + years ----
    y_train_raw = y[tr].astype(np.float32)
    y_test_raw  = y[te].astype(np.float32)

    years_train = df["Year"].to_numpy()[tr]
    years_test  = df["Year"].to_numpy()[te]

    # ---- TRAIN-only year means ----
    year_means = build_year_mean_map(years_train, y_train_raw)
    fallback = float(np.mean(y_train_raw))

    mu_train = lookup_year_means(years_train, year_means, fallback)
    mu_test  = lookup_year_means(years_test,  year_means, fallback)

    # ---- Normalized targets for training ----
    y_train = (y_train_raw / mu_train).astype(np.float32)
    y_test_ratio = (y_test_raw / mu_test).astype(np.float32)

    # ---- Model (CPU) ----
    model = HistGradientBoostingRegressor(
        max_depth=8,
        learning_rate=0.07,
        max_iter=300,
        random_state=42
    )

    model.fit(X_train, y_train)
    pred_ratio = model.predict(X_test).astype(np.float32)

    # ---- Back to raw for metrics ----
    pred_raw = pred_ratio * mu_test

    r2, rmse, mae = regression_metrics(y_test_raw, pred_raw)
    zacc, cmz = zone_metrics_pm10_from_ratio(y_test_ratio, pred_ratio)

    r2s.append(r2); rmses.append(rmse); maes.append(mae)
    zone_accs.append(zacc); cm_zone_sum += cmz

    fold_time = time.time() - fold_start
    fold_times.append(fold_time)

    avg_fold = float(np.mean(fold_times))
    eta_remaining = (n_folds - k) * avg_fold
    elapsed = time.time() - overall_start

    print(
        f"Fold {k}/{n_folds}: "
        f"R²={r2:.3f} | RMSE={rmse:.3f} | MAE={mae:.3f} || zone_acc={zacc:.3f} "
        f"| fold_time={fmt_time(fold_time)} | elapsed={fmt_time(elapsed)} | ETA={fmt_time(eta_remaining)}"
    )


summarize_cv_regression(r2s, rmses, maes, name="HistGradientBoostingRegressor (SAR + Year + latitude)")
summarize_cv_zones(zone_accs, cm_zone_sum, name="HistGradientBoostingRegressor (zone eval)")
print("Total wall time:", fmt_time(time.time() - overall_start))


Fold 1/5: R²=0.421 | RMSE=10.146 | MAE=7.862 || zone_acc=0.513 | fold_time=00m 32s | elapsed=00m 32s | ETA=02m 11s
Fold 2/5: R²=0.479 | RMSE=10.644 | MAE=7.930 || zone_acc=0.546 | fold_time=01m 11s | elapsed=01m 44s | ETA=02m 36s
Fold 3/5: R²=0.375 | RMSE=10.527 | MAE=7.964 || zone_acc=0.516 | fold_time=00m 29s | elapsed=02m 13s | ETA=01m 29s
Fold 4/5: R²=0.457 | RMSE=9.290 | MAE=7.168 || zone_acc=0.557 | fold_time=00m 57s | elapsed=03m 11s | ETA=00m 47s
Fold 5/5: R²=0.565 | RMSE=9.619 | MAE=7.413 || zone_acc=0.542 | fold_time=00m 39s | elapsed=03m 51s | ETA=00m 00s

HistGradientBoostingRegressor (SAR + Year + latitude) CV Summary (regression)
R²:   0.459 ± 0.064
RMSE: 10.045 ± 0.520
MAE:  7.667 ± 0.319

HistGradientBoostingRegressor (zone eval) CV Summary (zone-from-regression)
Zone Accuracy: 0.535 ± 0.017
Zone confusion matrix sum (rows=true low/med/high, cols=pred):
 [[104226  95975  10568]
 [ 51531 171438  25626]
 [ 17946 121350  95590]]
Total wall time: 03m 51s
